# 04 · Backtesting the strategy - with real costs

**Supports agenda block 7** ("Backtesting the Strategy") and sets up
block 9 ("Where to Go Next... why most backtests break"). We turn the
out-of-sample predictions from `03_model_training` into a monthly
top-N, equal-weight ETF portfolio and run it through `ml4t-backtest`'s
event-driven engine - **with commissions and slippage switched on
explicitly**.

That last clause is not boilerplate. `BacktestConfig` defaults
`commission_type` and `slippage_type` to `NONE` - a config that *sets*
`commission_rate`/`slippage_rate` without also setting the corresponding
`*_type` field runs at **zero cost**, silently. It's an easy mistake to
make (and one worth checking for explicitly if you ever hand this brief
to a coding agent) since the cost fields look like they should be enough
on their own.

In [1]:
from dataclasses import asdict

import pandas as pd
import polars as pl
from ml4t.backtest import (
    BacktestConfig,
    Broker,
    RebalanceConfig,
    Strategy,
    TargetWeightExecutor,
    run_backtest,
)
from ml4t.backtest.config import CommissionType, ShareType, SlippageType, SpreadConvention
from ml4t.backtest.execution.schedule import RebalanceCadence, RebalanceSchedule

DATA_DIR = "../data"

prices = pd.read_parquet(f"{DATA_DIR}/etf_universe.parquet")
prices["timestamp"] = pd.to_datetime(prices["timestamp"])

oos = pd.read_parquet(f"{DATA_DIR}/oos_predictions.parquet")
oos["timestamp"] = pd.to_datetime(oos["timestamp"]).dt.tz_localize(None)

# The engine wants prices covering exactly the OOS prediction window -
# trading on dates the model never produced a signal for isn't meaningful.
prices = prices[
    (prices["timestamp"] >= oos["timestamp"].min())
    & (prices["timestamp"] <= oos["timestamp"].max())
]

prices_pl = pl.from_pandas(
    prices[["timestamp", "symbol", "open", "high", "low", "close", "volume"]]
)
signals_pl = pl.from_pandas(oos[["timestamp", "symbol", "prediction"]])

print(f"backtest window: {prices['timestamp'].min().date()} -> {prices['timestamp'].max().date()}")
print(f"{prices_pl.height:,} price rows, {signals_pl.height:,} signal rows")

backtest window: 2009-01-02 -> 2025-01-10
388,755 price rows, 337,825 signal rows


## The cost model - matches the book's own ETF case study

`$0.0035`/share commission and tiered half-spread slippage are the same
numbers the case study's `config/setup.yaml` uses (`per_share_plus_spread`,
IBKR Pro tiered pricing). We use the real numbers rather than a round
"50 bps" placeholder, because the point of this block is that the *shape*
of a realistic cost model - fixed-plus-spread, not a flat percentage -
changes which trades are worth making at all, especially for the
less-liquid names in a 100-ETF universe.

In [2]:
ASSET_SPREADS = {
    "SPY": 0.005,
    "QQQ": 0.005,
    "IWM": 0.005,
    "EFA": 0.005,
    "EEM": 0.005,
    "DIA": 0.005,
    "VTI": 0.005,
    "XLK": 0.01,
    "XLF": 0.01,
    "XLV": 0.01,
    "XLE": 0.01,
    "XLY": 0.01,
    "XLI": 0.01,
    "XLP": 0.01,
    "XLU": 0.01,
    "XLB": 0.01,
    "XLRE": 0.01,
    "XLC": 0.01,
}

config = BacktestConfig(
    initial_cash=100_000.0,
    share_type=ShareType.INTEGER,
    commission_type=CommissionType.PER_SHARE,
    commission_per_share=0.0035,
    slippage_type=SlippageType.SPREAD,
    slippage_spread=0.02,  # default half-spread for anything not in the table below
    slippage_spread_by_asset=ASSET_SPREADS,
    slippage_spread_convention=SpreadConvention.HALF_SPREAD,
)
print(config.validate())  # empty list = no configuration warnings

[]


## The strategy: monthly top-10, equal weight

Rank the current cross-section by predicted 21-day forward return, take
the top 10, equal-weight them, rebalance at month end. `RebalanceConfig`
carries the same trade-filtering thresholds as the case study
(`min_weight_change=0.005`, `min_trade_value=$100`) so the backtest
doesn't churn on economically meaningless rebalances.

In [3]:
TOP_N = 10

executor = TargetWeightExecutor(
    config=RebalanceConfig(
        schedule=RebalanceSchedule(cadence=RebalanceCadence.MONTH_END),
        min_weight_change=0.005,
        min_trade_value=100.0,
    )
)


class TopNRebalanceStrategy(Strategy):
    def __init__(self, top_n: int, executor: TargetWeightExecutor):
        self.top_n = top_n
        self.executor = executor

    def on_prepare(self, broker: Broker, timestamps, config=None) -> None:
        self.executor.prepare_schedule(timestamps)

    def on_data(self, timestamp, data, context, broker: Broker) -> None:
        preds = {
            asset: bar["signals"]["prediction"]
            for asset, bar in data.items()
            if bar.get("signals") and bar["signals"].get("prediction") is not None
        }
        if not preds:
            return
        ranked = sorted(preds.items(), key=lambda kv: kv[1], reverse=True)[: self.top_n]
        weight = 1.0 / len(ranked)
        target_weights = {asset: weight for asset, _ in ranked}
        self.executor.execute(target_weights, data, broker, timestamp=timestamp)


strategy = TopNRebalanceStrategy(top_n=TOP_N, executor=executor)

## Run it twice: with and without costs

The only way to see what costs actually cost is to run the identical
strategy both ways and diff the result - not to quote a rule-of-thumb
bps haircut.

In [4]:
result_with_costs = run_backtest(prices_pl, strategy, signals=signals_pl, config=config)

zero_cost_config = BacktestConfig(initial_cash=100_000.0, share_type=ShareType.INTEGER)
strategy_zero_cost = TopNRebalanceStrategy(
    top_n=TOP_N,
    executor=TargetWeightExecutor(
        config=RebalanceConfig(
            schedule=RebalanceSchedule(cadence=RebalanceCadence.MONTH_END),
            min_weight_change=0.005,
            min_trade_value=100.0,
        )
    ),
)
result_zero_cost = run_backtest(
    prices_pl, strategy_zero_cost, signals=signals_pl, config=zero_cost_config
)

In [5]:
comparison = pd.DataFrame(
    {
        "zero_cost": result_zero_cost.metrics,
        "with_costs": result_with_costs.metrics,
    }
)
print(
    comparison.loc[
        ["total_return_pct", "sharpe", "max_drawdown_pct", "cagr", "avg_turnover", "total_costs"]
    ]
)

                   zero_cost    with_costs
total_return_pct  270.648208    218.298624
sharpe              0.529468      0.478774
max_drawdown_pct   38.844967     39.271995
cagr                0.085204      0.074940
avg_turnover        0.063809      0.063866
total_costs         0.000000  28917.679500


## Read this like a practitioner, not a scoreboard

`03_model_training` already showed a HAC-corrected IC that isn't
distinguishable from noise (t ≈ 0.65). Whatever this backtest reports
is the honest downstream consequence of that - not a separate, better
story. Two things worth checking regardless of the top-line number:

- **The cost gap.** `with_costs` vs `zero_cost` on the *same* signal,
  same rebalance dates, same ranking - the only thing that changed is
  whether commissions and spread are switched on. That gap is what
  "cost-aware" means in practice, not a modifier applied after the fact.
- **Turnover.** A monthly top-10-of-100 rebalance can still churn
  heavily if the ranking is unstable month to month; check
  `result_with_costs.trades` before trusting any Sharpe number here.

Run against the full 2009-2025 window, this strategy's Sharpe drops from
0.53 (zero cost) to 0.48 (with costs) - a real but modest haircut,
because monthly rebalancing keeps turnover low (~6.4% per rebalance).
The dollar cost is not modest: **$28,918 in commissions and spread on a
$100,000 starting account over 16 years** - money a zero-cost backtest
would have quietly kept for itself. A strategy with a weaker HAC IC or a
shorter rebalance horizon would show a much larger gap; this is close to
the best case for "costs don't matter much here."

In [6]:
trades = pd.DataFrame([asdict(t) for t in result_with_costs.trades])
print(f"trades (with costs): {len(trades)}")
trades.head()

trades (with costs): 1283


,symbol,entry_time,exit_time,entry_price,exit_price,quantity,pnl,pnl_percent,bars_held,fees,...,entry_bid_price,entry_ask_price,entry_spread,entry_available_size,exit_quote_mid_price,exit_bid_price,exit_ask_price,exit_spread,exit_available_size,metadata
0,USO,2009-02-02,2009-03-02,227.540004,204.300007,43.0,-999.620869,-0.102136,19,0.301,...,None,None,None,3504063.0,None,None,None,None,3972913.0,None
1,IJR,2009-02-02,2009-03-02,14.995912,13.084908,655.0,-1256.292847,-0.127435,19,4.585,...,None,None,None,3776600.0,None,None,None,None,5954400.0,None
2,EWA,2009-02-02,2009-03-02,5.515902,5.201346,1788.0,-574.941699,-0.057027,19,12.516,...,None,None,None,1818800.0,None,None,None,None,2602100.0,None
3,IBB,2009-02-02,2009-03-02,22.376521,19.752117,441.0,-1160.449119,-0.117284,19,3.087,...,None,None,None,3521100.0,None,None,None,None,14352600.0,None
4,XRT,2009-02-02,2009-03-02,7.705846,7.693575,1283.0,-24.724772,-0.001592,19,8.981,...,None,None,None,19567800.0,None,None,None,None,14400200.0,None


**This is where the pre-built notebooks stop.** Block 9 (research
agents) and block 10 (where to go next) are delivered live - see
`../docs/research_agent_demo.md` and the closing slides for what comes
after "the backtest looks weak, now what."